<a href="https://colab.research.google.com/github/rahu2004/ML-with-pyspark/blob/main/NLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pyspark

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

from pyspark.ml.feature import Tokenizer
from pyspark.ml.feature import StopWordsRemover
from pyspark.ml.feature import HashingTF
from pyspark.ml.feature import IDF
from pyspark.ml.feature import StringIndexer

from pyspark.ml.classification import LogisticRegression
from pyspark.ml import Pipeline

from pyspark.ml.evaluation import MulticlassClassificationEvaluator

In [3]:
spark = SparkSession.builder \
    .appName("PySparkNLP") \
    .getOrCreate()

sample dataset

In [20]:
data = [
    (0, "This product is amazing", "positive"),
    (1, "Worst experience ever", "negative"),
    (2, "I loved this item", "positive"),
    (3, "Very bad quality", "negative"),
    (4, "Fantastic purchase", "positive"),
    (5, "Not worth the money", "negative"),
    (6, "Excellent support", "positive"),
    (7, "Terrible customer service", "negative"),
    (8, "Highly recommended", "positive"),
    (9, "I hate this product", "negative"),
    (10, "Very satisfied", "positive"),
    (11, "Awful experience", "negative"),
    (12, "Great quality", "positive"),
    (13, "Poor packaging", "negative"),
    (14, "Amazing delivery speed", "positive"),
    (15, "Completely disappointed", "negative")

]

columns = ["id", "review", "sentiment"]

spark_df = spark.createDataFrame(data, columns)

In [21]:
spark_df.show()

+---+--------------------+---------+
| id|              review|sentiment|
+---+--------------------+---------+
|  0|This product is a...| positive|
|  1|Worst experience ...| negative|
|  2|   I loved this item| positive|
|  3|    Very bad quality| negative|
|  4|  Fantastic purchase| positive|
|  5| Not worth the money| negative|
|  6|   Excellent support| positive|
|  7|Terrible customer...| negative|
|  8|  Highly recommended| positive|
|  9| I hate this product| negative|
| 10|      Very satisfied| positive|
| 11|    Awful experience| negative|
| 12|       Great quality| positive|
| 13|      Poor packaging| negative|
| 14|Amazing delivery ...| positive|
| 15|Completely disapp...| negative|
+---+--------------------+---------+



In [22]:
label_indexer = StringIndexer(
    inputCol='sentiment',
    outputCol='label'
)

In [23]:
tokenizer = Tokenizer(
    inputCol='review',
    outputCol='words'
)

In [24]:
stopword_remover = StopWordsRemover(
    inputCol='words',
    outputCol='filtered_words'
)

In [25]:
hashing_tf = HashingTF(
    inputCol='filtered_words',
    outputCol='raw_features',
    numFeatures=1000
)

In [26]:
idf = IDF(
    inputCol='raw_features',
    outputCol='features'
)

In [27]:
lr = LogisticRegression(
    featuresCol='features',
    labelCol='label'
)

In [28]:
pipeline = Pipeline(stages=[
    label_indexer,
    tokenizer,
    stopword_remover,
    hashing_tf,
    idf,
    lr
])

In [29]:
train_data, test_data = spark_df.randomSplit([0.9, 0.1], seed=42)


pipeline_model = pipeline.fit(train_data)

In [30]:
predictions = pipeline_model.transform(test_data)

In [31]:
predictions.select(
    'review',
    'sentiment',
    'prediction'
).show(truncate=False)

+-----------------+---------+----------+
|review           |sentiment|prediction|
+-----------------+---------+----------+
|Excellent support|positive |1.0       |
+-----------------+---------+----------+



In [32]:
evaluator = MulticlassClassificationEvaluator(
    labelCol='label',
    predictionCol='prediction',
    metricName='accuracy'
)

accuracy = evaluator.evaluate(predictions)

print("Accuracy:", accuracy)

Accuracy: 1.0


In [33]:
new_reviews = [
    (8, "Excellent product with great quality", "positive"),
    (9, "I hate this product", "negative")
]

new_df = spark.createDataFrame(new_reviews, columns)

custom_predictions = pipeline_model.transform(new_df)

custom_predictions.select(
    'review',
    'prediction'
).show(truncate=False)

+------------------------------------+----------+
|review                              |prediction|
+------------------------------------+----------+
|Excellent product with great quality|1.0       |
|I hate this product                 |0.0       |
+------------------------------------+----------+

